# 🎬 Hybrid Movie Recommendation + Fine-Grained Sentiment Analysis
### Complete 15-Stage Google Colab Project

This notebook implements all 15 stages of the given movie recommendation, review sentiment and LLM-assistant project.


In [ ]:
!pip -q install textblob

from textblob import TextBlob
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML


In [ ]:
# Sample movie metadata
movies = pd.DataFrame([
    [1,"Interstellar","Sci-Fi Adventure","space science future family emotional",8.7],
    [2,"Inception","Sci-Fi Thriller","dream mind heist action mystery",8.8],
    [3,"The Dark Knight","Action Crime","superhero crime action thriller intense",9.0],
    [4,"La La Land","Romance Musical","romance music love emotional artistic",8.0],
    [5,"The Martian","Sci-Fi Adventure","space survival science humor",8.0],
    [6,"Toy Story","Animation Family","animation friendship family comedy",8.3],
    [7,"Coco","Animation Family","music family animation emotional culture",8.4],
    [8,"The Notebook","Romance Drama","romance love emotional relationship",7.8],
    [9,"Avengers: Endgame","Action Sci-Fi","superhero action adventure friendship emotional",8.4],
    [10,"The Grand Budapest Hotel","Comedy Drama","comedy artistic adventure quirky",8.1]
], columns=["movie_id","title","genre","tags","rating"])

# Sample user reviews with fine-grained aspects
reviews = pd.DataFrame([
    ["Interstellar","The visuals are amazing and the science is fascinating.","positive","visuals"],
    ["Interstellar","The family story is touching and emotional.","positive","family"],
    ["Interstellar","The middle part feels a little slow.","negative","pacing"],
    ["Inception","Very clever story with excellent mind-bending ideas.","positive","story"],
    ["Inception","The action and visuals are impressive.","positive","visuals"],
    ["Inception","Some parts are confusing on the first watch.","negative","clarity"],
    ["The Dark Knight","Great acting and intense action.","positive","acting"],
    ["The Dark Knight","The villain is outstanding.","positive","acting"],
    ["The Dark Knight","A few scenes are quite dark.","negative","visuals"],
    ["La La Land","Beautiful music and wonderful performances.","positive","music"],
    ["La La Land","The romance is emotional and memorable.","positive","romance"],
    ["La La Land","The ending may feel sad.","negative","ending"],
    ["The Martian","Funny, smart and inspiring survival story.","positive","story"],
    ["The Martian","The science details are interesting.","positive","science"],
    ["The Martian","Some technical scenes are lengthy.","negative","pacing"],
    ["Toy Story","Fun, warm and perfect for families.","positive","family"],
    ["Toy Story","The characters are lovable.","positive","characters"],
    ["Coco","Beautiful music and a touching family story.","positive","music"],
    ["Coco","Very emotional and visually beautiful.","positive","visuals"],
    ["The Notebook","A romantic and emotional love story.","positive","romance"],
    ["The Notebook","It is quite predictable.","negative","story"],
    ["Avengers: Endgame","Amazing action and emotional character moments.","positive","action"],
    ["Avengers: Endgame","Great payoff for superhero fans.","positive","story"],
    ["The Grand Budapest Hotel","Funny, stylish and visually unique.","positive","visuals"],
    ["The Grand Budapest Hotel","The quirky characters are entertaining.","positive","characters"]
], columns=["title","review","sentiment","aspect"])

display(movies)
display(reviews.head())


## 1. Develop the Movie Recommendation Model

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
movie_matrix = vectorizer.fit_transform(movies["genre"] + " " + movies["tags"])
similarity = cosine_similarity(movie_matrix)

def recommend_by_movie(movie_title, n=5):
    idx = movies.index[movies["title"].str.lower() == movie_title.lower()]
    if len(idx) == 0:
        return pd.DataFrame(columns=["title","genre","rating","similarity"])
    i = idx[0]
    scores = sorted(enumerate(similarity[i]), key=lambda x:x[1], reverse=True)
    scores = [(j,s) for j,s in scores if j != i][:n]
    result = movies.iloc[[j for j,_ in scores]][["title","genre","rating"]].copy()
    result["similarity"] = [round(s,3) for _,s in scores]
    return result

display(recommend_by_movie("Interstellar"))


## 2. Connect Recommendations with Review Sentiment

In [ ]:
def review_summary(title):
    r = reviews[reviews["title"] == title]
    pos = r[r["sentiment"]=="positive"]
    neg = r[r["sentiment"]=="negative"]
    return {
        "positive_reviews": len(pos),
        "negative_reviews": len(neg),
        "positive_aspects": ", ".join(pos["aspect"].unique()),
        "negative_aspects": ", ".join(neg["aspect"].unique())
    }

rec = recommend_by_movie("Interstellar")
rows = []
for title in rec["title"]:
    s = review_summary(title)
    rows.append([title,s["positive_reviews"],s["negative_reviews"],
                 s["positive_aspects"],s["negative_aspects"]])
connected = pd.DataFrame(rows, columns=["title","positive_reviews","negative_reviews",
                                         "positive_aspects","negative_aspects"])
display(connected)


## 3. Prepare Movie Metadata and Review Information for Retrieval

In [ ]:
retrieval = movies.copy()
retrieval["review_text"] = retrieval["title"].apply(
    lambda x: " ".join(reviews.loc[reviews["title"]==x,"review"].tolist())
)
retrieval["pros"] = retrieval["title"].apply(
    lambda x: ", ".join(reviews.loc[(reviews["title"]==x)&(reviews["sentiment"]=="positive"),"aspect"].unique())
)
retrieval["cons"] = retrieval["title"].apply(
    lambda x: ", ".join(reviews.loc[(reviews["title"]==x)&(reviews["sentiment"]=="negative"),"aspect"].unique())
)
display(retrieval[["title","genre","tags","review_text","pros","cons"]])


## 4. Develop Fine-Grained Sentiment Classifier

In [ ]:
def fine_grained_sentiment(review):
    polarity = TextBlob(review).sentiment.polarity
    if polarity > 0.15:
        label = "Positive"
    elif polarity < -0.15:
        label = "Negative"
    else:
        label = "Neutral"

    aspect_words = {
        "story":["story","plot","ending"],
        "acting":["acting","performance","performances","characters","villain"],
        "visuals":["visuals","visual","style"],
        "music":["music"],
        "pacing":["slow","lengthy","pacing"],
        "family":["family"],
        "romance":["romance","love"],
        "science":["science"],
        "action":["action"]
    }
    aspect = "general"
    low = review.lower()
    for a, words in aspect_words.items():
        if any(w in low for w in words):
            aspect = a
            break
    return aspect, label, round(polarity,3)

sample = "The visuals are beautiful but the story is a little slow."
print(fine_grained_sentiment(sample))


## 5. Design Prompts for Movie Summaries and Pros/Cons

In [ ]:
summary_prompt = '''
You are a movie recommendation assistant.
Movie: {movie}
Genre: {genre}
Rating: {rating}
Pros: {pros}
Cons: {cons}

Give:
1. A short movie summary.
2. 3 key pros.
3. 2 key cons.
4. Who should watch it.
Use simple language and do not invent facts.
'''
print(summary_prompt)


## 6. Develop the Movie Recommendation Assistant

In [ ]:
def movie_assistant(movie_title, n=5):
    rec = recommend_by_movie(movie_title,n)
    if rec.empty:
        print("Movie not found.")
        return
    rows=[]
    for _,row in rec.iterrows():
        s=review_summary(row["title"])
        rows.append([row["title"],row["genre"],row["rating"],row["similarity"],
                     s["positive_aspects"],s["negative_aspects"]])
    display(pd.DataFrame(rows,columns=["Movie","Genre","Rating","Similarity","Pros","Cons"]))

movie_assistant("Interstellar")


## 7. Test Recommendations Using Different Movie Preferences

In [ ]:
test_preferences = ["Interstellar","La La Land","Toy Story","The Dark Knight"]

for preference in test_preferences:
    print("\nPreference:",preference)
    display(recommend_by_movie(preference,3))


## 8. Evaluate Recommendation Relevance and Summary Quality

In [ ]:
evaluation=[]
for title in test_preferences:
    r=recommend_by_movie(title,3)
    avg_sim=round(r["similarity"].mean(),3) if not r.empty else 0
    coverage=round(r["title"].isin(retrieval["title"]).mean(),3) if not r.empty else 0
    evaluation.append([title,avg_sim,coverage])

evaluation_df=pd.DataFrame(evaluation,
    columns=["Preference","Average Similarity","Review Retrieval Coverage"])
display(evaluation_df)


## 9. Identify Inconsistent Sentiment or Recommendations

In [ ]:
checks=[]
for _,row in reviews.iterrows():
    _,predicted,polarity=fine_grained_sentiment(row["review"])
    stored=row["sentiment"].capitalize()
    if predicted != stored and predicted != "Neutral":
        checks.append([row["title"],row["review"],stored,predicted,polarity])

inconsistent=pd.DataFrame(checks,
    columns=["Movie","Review","Stored","Predicted","Polarity"])
if inconsistent.empty:
    print("No strong sentiment inconsistencies found.")
else:
    display(inconsistent)


## 10. Improve Recommendation Prompts

In [ ]:
improved_prompt = '''
Recommend movies for this user preference: {preference}

Use only the retrieved information:
Title: {title}
Genre: {genre}
Rating: {rating}
Similarity: {similarity}
Pros from reviews: {pros}
Cons from reviews: {cons}

Explain why the movie matches the user's preference.
Do not invent actors, story details, ratings or review information.
'''
print(improved_prompt)


## 11. Add Personalized Recommendation Explanations

In [ ]:
def personalized_explanation(preference,movie_title,similarity_score):
    movie=movies[movies["title"]==movie_title].iloc[0]
    s=review_summary(movie_title)
    return (
        f"{movie_title} is recommended because it matches your preference for "
        f"{preference}. Similarity score: {similarity_score:.2f}. "
        f"Review pros: {s['positive_aspects'] or 'general positive feedback'}. "
        f"Main concern: {s['negative_aspects'] or 'no major concern recorded'}."
    )

r=recommend_by_movie("Interstellar",3)
for _,row in r.iterrows():
    print(personalized_explanation("science fiction",row["title"],row["similarity"]))


## 12. Develop Movie Assistant Interface

In [ ]:
movie_options="".join(
    f'<option value="{t}">{t}</option>' for t in movies["title"]
)

html=f'''
<div style="font-family:Arial;padding:20px;border:1px solid #ccc;border-radius:12px;">
<h2>🎬 Movie Recommendation Assistant</h2>
<p>Select a movie:</p>
<select id="movieSelect" style="padding:8px;">{movie_options}</select>
<button onclick="showMessage()" style="padding:8px 14px;">Recommend</button>
<p id="output"></p>
<script>
function showMessage(){{
let movie=document.getElementById("movieSelect").value;
document.getElementById("output").innerHTML="Selected preference: <b>"+movie+
"</b><br>Use the notebook assistant function to see recommendations.";
}}
</script>
</div>
'''
display(HTML(html))


## 13. Integrate Recommender, Sentiment and LLM

In [ ]:
def integrated_assistant(movie_title,user_preference="general"):
    rec=recommend_by_movie(movie_title,5)
    outputs=[]
    for _,row in rec.iterrows():
        s=review_summary(row["title"])
        prompt=improved_prompt.format(
            preference=user_preference,title=row["title"],genre=row["genre"],
            rating=row["rating"],similarity=round(row["similarity"],3),
            pros=s["positive_aspects"],cons=s["negative_aspects"])
        outputs.append({
            "movie":row["title"],
            "similarity":round(row["similarity"],3),
            "pros":s["positive_aspects"],
            "cons":s["negative_aspects"],
            "llm_prompt":prompt
        })
    return outputs

integrated=integrated_assistant(
    "Interstellar","science fiction and emotional movies")
for item in integrated[:2]:
    print("="*70)
    print(item["movie"])
    print("Pros:",item["pros"])
    print("Cons:",item["cons"])
    print("\nLLM Prompt:\n",item["llm_prompt"])


## 14. Validate New Movie Preferences

In [ ]:
def preference_search(preference,n=5):
    pref_vec=vectorizer.transform([preference])
    scores=cosine_similarity(pref_vec,movie_matrix).flatten()
    order=np.argsort(scores)[::-1][:n]
    result=movies.iloc[order][["title","genre","rating"]].copy()
    result["preference_score"]=np.round(scores[order],3)
    return result

new_preferences=[
    "emotional family movies with music",
    "smart science fiction with survival",
    "intense superhero action",
    "romantic movies with beautiful music"
]

for p in new_preferences:
    print("\nNew preference:",p)
    display(preference_search(p))


## 15. Demonstrate the Movie Assistant

In [ ]:
def demonstrate_movie_assistant(preference,n=5):
    print("USER PREFERENCE:",preference)
    result=preference_search(preference,n)
    final_rows=[]
    for _,row in result.iterrows():
        s=review_summary(row["title"])
        explanation=personalized_explanation(
            preference,row["title"],row["preference_score"])
        final_rows.append([
            row["title"],row["genre"],row["rating"],row["preference_score"],
            s["positive_aspects"],s["negative_aspects"],explanation
        ])
    final_df=pd.DataFrame(final_rows,columns=[
        "Movie","Genre","Rating","Score","Pros","Cons","Personalized Explanation"])
    display(final_df)

demonstrate_movie_assistant(
    "science fiction with science, adventure and emotional moments")


# ✅ Final Result

The notebook implements all 15 requested stages:
1. Movie recommendation model
2. Recommendation + review sentiment connection
3. Movie metadata and review retrieval
4. Fine-grained sentiment classifier
5. Summary and Pros/Cons prompts
6. Movie recommendation assistant
7. Preference testing
8. Recommendation and summary evaluation
9. Inconsistency detection
10. Prompt improvement
11. Personalized explanations
12. Movie assistant interface
13. Recommender + sentiment + LLM prompt integration
14. New preference validation
15. Final demonstration

**Important:** This is a self-contained Colab prototype. No API key is required. Stage 13 prepares grounded prompts for an LLM; an actual Gemini/OpenAI API can be connected later.
